# 3D Diffusion Model for Scalar Field Theory

This notebook demonstrates how to use the 3D diffusion model for lattice field theory simulations.


In [1]:
import os
import numpy as np
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt

from diffusion_lightning_3d import (
    DiffusionModel3D, FieldDataModule3D, 
    grab, get_mag_3d, get_abs_mag_3d, get_chi2_3d, get_UL_3d, mag_3d
)

# Create directories
for folder in ['data', 'figures', 'models']:
    os.makedirs(folder, exist_ok=True)

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")


Using GPU: NVIDIA GeForce RTX 4090


## 1. Configuration


In [2]:
# Lattice parameters
L = 32          # 3D lattice size: L x L x L (use smaller L for 3D due to memory)
k = 0.14         # Hopping parameter
l = 2.5       # Coupling
sigma = 25.0    # SDE noise scale

# Training parameters
n_epochs = 250
batch_size = 64  # Smaller batch size for 3D due to memory constraints
lr = 1e-3

# Sampling parameters
num_steps = 1000
num_samples = 256  # Fewer samples for 3D

# Data path (adjust to your 3D data)
data_path = f"data/cfgs_k={k}_l={l}_32^3.jld2"


## 2. Prepare Data (Optional: Generate Synthetic 3D Data)


In [3]:
# Generate synthetic 3D data for testing (uncomment to use)
def generate_synthetic_3d_data(n_samples, L, seed=42):
    """Generate synthetic 3D field configurations for testing."""
    np.random.seed(seed)
    cfgs = np.zeros((n_samples, L, L, L), dtype=np.float32)
    
    for i in range(n_samples):
        # Start with random noise
        field = np.random.randn(L, L, L).astype(np.float32)
        # Apply some smoothing to create correlations
        from scipy.ndimage import gaussian_filter
        field = gaussian_filter(field, sigma=1.5)
        field = field * 2.0
        cfgs[i] = field
    
    return cfgs

# Uncomment to generate and save synthetic data:
# import h5py
# synthetic_cfgs = generate_synthetic_3d_data(n_samples=10000, L=L)
# with h5py.File(data_path, 'w') as f:
#     f.create_dataset('cfgs', data=synthetic_cfgs)
# print(f"Generated {synthetic_cfgs.shape[0]} synthetic 3D configurations")


## 3. Create Model


In [3]:
# Create the 3D diffusion model with MEMORY OPTIMIZATION
model = DiffusionModel3D(
    sigma=sigma,
    lr=lr,
    L=L,
    periodic=True,           # Use periodic boundary conditions
    symmetry_aug=False,       # Apply symmetry augmentations during training
    z2_arch=False,           # Z2 symmetric architecture (optional)
    cubic_arch=False,        # Cubic equivariant architecture (expensive: 24x compute)
    full_cubic_group=False,  # Use full Oh group (48 elements) vs O group (24)
    # ===== MEMORY OPTIMIZATION OPTIONS =====
    use_checkpoint=True,     # Enable gradient checkpointing (~30-50% memory savings)
)

# Print model summary
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Gradient checkpointing: {model.use_checkpoint}")


Model parameters: 2,849,761
Gradient checkpointing: True


## 4. Train Model


In [4]:
# Set up data module (uncomment when you have data)
data_module = FieldDataModule3D(
    data_path=data_path,
    batch_size=batch_size,
    normalize=True
)
data_module.setup()
print(f"Training samples: {len(data_module.train_data)}")

# Set up training
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath='models',
    filename=f'diffusion_L{L}_k{k}_l{l}_32^3',
    save_top_k=3,
    monitor='train_loss',
    mode='min'
)

# ===== MEMORY OPTIMIZATION IN TRAINER =====
# 1. precision="16-mixed" or "bf16-mixed": Mixed precision training (~50% memory savings)
# 2. accumulate_grad_batches: Simulate larger batches without more memory
# 3. gradient_clip_val: Prevent gradient explosions with AMP

trainer = pl.Trainer(
    max_epochs=n_epochs,
    accelerator='gpu',
    devices=[2],
    callbacks=[checkpoint_callback],
    enable_progress_bar=True,
    log_every_n_steps=50,
    # ===== MEMORY OPTIMIZATION OPTIONS =====
    precision="16-mixed",        # Use AMP (float16) - ~50% memory savings!
    accumulate_grad_batches=4,   # Effective batch = batch_size * 4 = 128
    gradient_clip_val=1.0,       # Recommended with AMP to prevent gradient explosions
)

print(f"Training with precision: {trainer.precision}")
print(f"Gradient accumulation: {trainer.accumulate_grad_batches} steps")
print(f"Effective batch size: {batch_size * trainer.accumulate_grad_batches}")

# Train the model
trainer.fit(model, data_module)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 5090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Training samples: 5120
Training with precision: 16-mixed
Gradient accumulation: 4 steps
Effective batch size: 256


/home/tyy/anaconda3/envs/nenv2/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /home/tyy/DM/DMasSQ-main/models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5]
/home/tyy/anaconda3/envs/nenv2/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name        | Type       | Params | Mode 
---------------------------------------------------
0 | score_model | ScoreNet3D | 2.8 M  | train
---------------------------------------------------
2.8 M     Trainable params
128       Non-trainable params
2.8 M     Total params
11.399    Total estimated model params size (MB)
33        Modules in train mode
0         Modules in eval mode
/home/tyy/anaconda3/envs/nenv2/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/tyy/anaconda3/envs/nenv2/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 5. Generate Samples


In [ ]:
# Move model to GPU if available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Generate samples with memory optimization
num_samples_generate = 64
sample_batch_size = 32

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=100,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=True,                # Use AMP during sampling (~30-50% faster, less memory)
    amp_dtype=torch.float16,     # or torch.bfloat16 for newer GPUs
)

print(f"Generated samples shape: {samples.shape}")


## 6. Visualize Samples


In [ ]:
# Visualize 3D samples using slices
samples_np = grab(samples[:, 0, :, :, :])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, ax in enumerate(axes.flat):
    if idx < len(samples_np):
        # Show middle slice in z-direction
        mid_z = samples_np.shape[3] // 2
        im = ax.imshow(samples_np[idx, :, :, mid_z], cmap='RdBu_r')
        ax.set_title(f'Sample {idx+1} (z={mid_z})')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Generated 3D Field Configurations (Middle Z-Slice)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize multiple slices of a single sample
sample_idx = 0
sample = samples_np[sample_idx]

n_slices = 4
slice_indices = np.linspace(0, sample.shape[2]-1, n_slices, dtype=int)

fig, axes = plt.subplots(1, n_slices, figsize=(16, 4))

for i, z_idx in enumerate(slice_indices):
    im = axes[i].imshow(sample[:, :, z_idx], cmap='RdBu_r')
    axes[i].set_title(f'z = {z_idx}')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.suptitle(f'Sample {sample_idx+1}: Z-Slices Through 3D Volume', fontsize=14)
plt.tight_layout()
plt.show()


## 7. Compute Observables


In [ ]:
# Compute magnetization
mag_mean, mag_err = get_mag_3d(samples_np)
abs_mag_mean, abs_mag_err = get_abs_mag_3d(samples_np)

print(f"Magnetization: {mag_mean:.6f} ± {mag_err:.6f}")
print(f"Absolute Magnetization: {abs_mag_mean:.6f} ± {abs_mag_err:.6f}")


In [ ]:
# Histogram of per-sample magnetization
mags = mag_3d(samples_np)

plt.figure(figsize=(10, 5))
plt.hist(mags, bins=30, density=True, alpha=0.7, color='steelblue', edgecolor='black')
plt.xlabel('Magnetization', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title('Distribution of Per-Sample Magnetization', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()


## 8. Memory Optimization Techniques

3D models require significantly more memory than 2D:
- A 2D field of size L×L has L² sites
- A 3D field of size L×L×L has L³ sites

For example, with L=32:
- 2D: 1,024 sites
- 3D: 32,768 sites (32× more!)

### Memory Optimization Options (without changing hyperparameters)

| Technique | Memory Savings | Speed Impact | How to Enable |
|-----------|----------------|--------------|---------------|
| **Mixed Precision (AMP)** | ~50% | ~10-30% faster | `precision="16-mixed"` in Trainer |
| **Gradient Checkpointing** | ~30-50% | ~20-30% slower | `use_checkpoint=True` in model |
| **Gradient Accumulation** | Proportional to steps | No impact | `accumulate_grad_batches=N` in Trainer |
| **AMP during Sampling** | ~30% | ~20% faster | `use_amp=True` in sample() |

### Combined Example:
```python
# Model with checkpointing
model = DiffusionModel3D(..., use_checkpoint=True)

# Trainer with AMP + accumulation
trainer = pl.Trainer(
    precision="16-mixed",          # AMP
    accumulate_grad_batches=4,     # Effective batch = 4x
    gradient_clip_val=1.0,         # Stability with AMP
)

# Sampling with AMP
samples = model.sample(..., use_amp=True)
```

With all optimizations, you can reduce memory by **~60-70%** while keeping the same hyperparameters!


In [ ]:
# Memory estimation with optimization comparison
def estimate_memory_mb(L, batch_size, channels=[32, 64, 128, 256], dtype_bytes=4, checkpoint=False):
    """Rough estimate of GPU memory needed for a single forward pass."""
    volume = L ** 3
    io_mem = batch_size * 1 * volume * dtype_bytes * 2
    feature_mem = 0
    for c in channels:
        feature_mem += batch_size * c * volume * dtype_bytes * 2
    total_mb = (io_mem + feature_mem) / (1024 ** 2)
    
    # Checkpointing roughly halves activation memory
    if checkpoint:
        total_mb *= 0.5
    
    return total_mb

print("Memory Comparison: FP32 vs FP16 (AMP) vs Checkpointing")
print("=" * 70)
print(f"{'L':>3} | {'Batch':>5} | {'FP32':>10} | {'FP16 (AMP)':>12} | {'FP32+Ckpt':>12} | {'FP16+Ckpt':>12}")
print("-" * 70)

for L_test in [16, 24, 32]:
    for bs_test in [32, 64]:
        mem_fp32 = estimate_memory_mb(L_test, bs_test, dtype_bytes=4)
        mem_fp16 = estimate_memory_mb(L_test, bs_test, dtype_bytes=2)
        mem_fp32_ckpt = estimate_memory_mb(L_test, bs_test, dtype_bytes=4, checkpoint=True)
        mem_fp16_ckpt = estimate_memory_mb(L_test, bs_test, dtype_bytes=2, checkpoint=True)
        print(f"{L_test:>3} | {bs_test:>5} | {mem_fp32:>8,.0f} MB | {mem_fp16:>10,.0f} MB | {mem_fp32_ckpt:>10,.0f} MB | {mem_fp16_ckpt:>10,.0f} MB")

print("\n💡 TIP: FP16 + Checkpointing can reduce memory by ~75%!")
